
# InterviewSense — Corrected Communication-Assessment Modeling Notebook

This notebook is the corrected **supervised modeling phase** for InterviewSense.

It uses the Kaggle **Multimodal English Speech Dataset** as the communication/speaking-assessment dataset. The dataset contains speech-duration, word-count, speaking-rate, pause, pronunciation, fluency, intonation and overall assessment information, plus a `Low / Medium / High` proficiency label.

## Important anti-leakage rule

The model must **not** use `pronunciation_score`, `fluency_score`, `intonation_score`, or `target_level` as input features when predicting `overall_score`, because those fields are direct assessment/label information.

The main model therefore learns from measurable speech-behavior variables such as:
- speech duration
- word count
- speaking rate
- pause count
- derived pause density
- derived speech activity

We train an interpretable linear model to obtain actual feature weights, and an independent tree-based model as the cross-validation/benchmark model.

Evaluation is speaker-grouped so recordings from the same speaker cannot leak across folds.


In [ ]:

# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
%pip install -q kagglehub pandas numpy scikit-learn joblib matplotlib


In [ ]:

# ============================================================
# 2. IMPORTS AND REPRODUCIBILITY
# ============================================================
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import kagglehub

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
    classification_report,
)

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("Working directory:", os.getcwd())


In [ ]:

# ============================================================
# 3. DOWNLOAD THE CORRECT KAGGLE DATASET
# ============================================================
DATASET_HANDLE = "colabsss/multimodal-english-speech-dataset"

DATASET_PATH = kagglehub.dataset_download(DATASET_HANDLE)
DATASET_PATH = Path(DATASET_PATH)

print("Dataset path:")
print(DATASET_PATH)


In [ ]:

# ============================================================
# 4. LOCATE THE DATASET CSV FILES
# ============================================================
csv_files = list(DATASET_PATH.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {DATASET_PATH}")

print("CSV files found:")
for p in csv_files:
    print(" -", p)


In [ ]:

# ============================================================
# 5. LOAD THE AUDIO SPEAKING DATASET
# ============================================================
audio_csv = None

for p in csv_files:
    if "audio" in p.name.lower():
        audio_csv = p
        break

if audio_csv is None:
    # Fallback: select the CSV containing the expected speaking columns.
    for p in csv_files:
        temp = pd.read_csv(p, nrows=5)
        cols = {str(c).lower() for c in temp.columns}
        if "speaking_rate_wpm" in cols and "overall_score" in cols:
            audio_csv = p
            break

if audio_csv is None:
    raise FileNotFoundError(
        "Could not identify the audio speaking CSV automatically. "
        "Inspect the printed CSV list and set audio_csv manually."
    )

audio_df = pd.read_csv(audio_csv)

print("Selected file:", audio_csv)
print("Shape:", audio_df.shape)
print("Columns:")
print(audio_df.columns.tolist())
display(audio_df.head())


In [ ]:

# ============================================================
# 6. VALIDATE THE EXPECTED SCHEMA
# ============================================================
EXPECTED_COLUMNS = [
    "sample_id",
    "speaker_id",
    "age",
    "gender",
    "region",
    "education_level",
    "english_learning_years",
    "task_type",
    "speech_duration_sec",
    "word_count",
    "speaking_rate_wpm",
    "pause_count",
    "pronunciation_score",
    "fluency_score",
    "intonation_score",
    "overall_score",
    "target_level",
]

missing = [c for c in EXPECTED_COLUMNS if c not in audio_df.columns]

if missing:
    raise ValueError(
        "The downloaded dataset is missing expected columns: "
        + ", ".join(missing)
    )

print("Schema validation PASSED.")
print("Rows:", len(audio_df))
print("Unique speakers:", audio_df["speaker_id"].nunique())
print("Target levels:")
print(audio_df["target_level"].value_counts(dropna=False))



## 7. Define the modeling features

For the main regression target (`overall_score`), the following are deliberately excluded:

- `pronunciation_score`
- `fluency_score`
- `intonation_score`
- `target_level`

These are assessment outputs and would leak the answer into the model.

Demographic variables are also excluded from the main communication model because the goal is to model speech performance rather than speaker identity or demographic characteristics.


In [ ]:

# ============================================================
# 8. EXACT INPUT FEATURES
# ============================================================
BASE_FEATURES = [
    "speech_duration_sec",
    "word_count",
    "speaking_rate_wpm",
    "pause_count",
]

TARGET = "overall_score"
GROUP = "speaker_id"

LEAKAGE_COLUMNS = [
    "pronunciation_score",
    "fluency_score",
    "intonation_score",
    "overall_score",
    "target_level",
]

print("Base features:")
for c in BASE_FEATURES:
    print(" -", c)

print("\nExcluded assessment/leakage columns:")
for c in LEAKAGE_COLUMNS:
    print(" -", c)


In [ ]:

# ============================================================
# 9. CREATE DERIVED SPEECH-BEHAVIOR FEATURES
# ============================================================
df = audio_df.copy()

df["speech_duration_sec"] = pd.to_numeric(df["speech_duration_sec"], errors="coerce")
df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce")
df["speaking_rate_wpm"] = pd.to_numeric(df["speaking_rate_wpm"], errors="coerce")
df["pause_count"] = pd.to_numeric(df["pause_count"], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

# Independently recompute WPM from word count and duration.
df["computed_wpm"] = np.where(
    df["speech_duration_sec"] > 0,
    df["word_count"] / (df["speech_duration_sec"] / 60.0),
    np.nan,
)

# Pause density normalizes pause count by speaking duration.
df["pause_density_per_min"] = np.where(
    df["speech_duration_sec"] > 0,
    df["pause_count"] / (df["speech_duration_sec"] / 60.0),
    np.nan,
)

# Word production per second is another equivalent but interpretable measure.
df["words_per_second"] = np.where(
    df["speech_duration_sec"] > 0,
    df["word_count"] / df["speech_duration_sec"],
    np.nan,
)

# Use only these measurable speech variables.
MODEL_FEATURES = [
    "speech_duration_sec",
    "word_count",
    "speaking_rate_wpm",
    "pause_count",
    "pause_density_per_min",
    "words_per_second",
]

model_df = df[
    [GROUP, TARGET] + MODEL_FEATURES + ["target_level"]
].copy()

model_df = model_df.dropna(subset=[GROUP, TARGET]).reset_index(drop=True)

for c in MODEL_FEATURES:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

model_df = model_df.replace([np.inf, -np.inf], np.nan)

print("Modeling rows:", len(model_df))
print("Modeling features:", MODEL_FEATURES)
print("Target range:", model_df[TARGET].min(), "->", model_df[TARGET].max())
print("Speakers:", model_df[GROUP].nunique())


In [ ]:

# ============================================================
# 10. DATA QUALITY CHECK
# ============================================================
quality = pd.DataFrame({
    "missing": model_df[MODEL_FEATURES + [TARGET]].isna().sum(),
    "dtype": model_df[MODEL_FEATURES + [TARGET]].dtypes.astype(str),
})

display(quality)

print("\nProvided WPM vs independently computed WPM:")
wpm_check = df[["speaking_rate_wpm", "computed_wpm"]].dropna()
wpm_check["absolute_difference"] = (
    wpm_check["speaking_rate_wpm"] - wpm_check["computed_wpm"]
).abs()

print(wpm_check["absolute_difference"].describe())

print("\nTarget distribution:")
display(model_df[TARGET].describe())


In [ ]:

# ============================================================
# 11. SPEAKER-INDEPENDENT TRAIN/TEST SPLIT
# ============================================================
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_SEED,
)

train_idx, test_idx = next(
    gss.split(
        model_df[MODEL_FEATURES],
        model_df[TARGET],
        groups=model_df[GROUP],
    )
)

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

train_speakers = set(train_df[GROUP].astype(str))
test_speakers = set(test_df[GROUP].astype(str))

assert train_speakers.isdisjoint(test_speakers)

X_train = train_df[MODEL_FEATURES]
y_train = train_df[TARGET]

X_test = test_df[MODEL_FEATURES]
y_test = test_df[TARGET]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train speakers:", len(train_speakers))
print("Test speakers:", len(test_speakers))
print("Speaker leakage:", len(train_speakers.intersection(test_speakers)))


In [ ]:

# ============================================================
# 12. PREPROCESSING PIPELINE
# ============================================================
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# The input features are all numeric, so one clean pipeline is sufficient.
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, MODEL_FEATURES),
])

print("Preprocessing pipeline ready.")



## 13. Main interpretable model — ElasticNet regression

This model is useful because it gives us **actual learned coefficients/weights**.

The pipeline standardizes the features first. Therefore, the coefficients are standardized-feature weights and are directly comparable in magnitude.

The model is evaluated with **GroupKFold**, not ordinary random K-fold, so one speaker cannot appear in both training and validation folds.


In [ ]:

# ============================================================
# 13. ELASTIC NET — INTERPRETABLE WEIGHTS
# ============================================================
elastic_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", ElasticNet(
        alpha=0.01,
        l1_ratio=0.20,
        max_iter=10000,
        random_state=RANDOM_SEED,
    )),
])

n_splits = min(5, train_df[GROUP].nunique())

if n_splits < 3:
    raise ValueError(
        f"Need at least 3 unique training speakers for grouped CV; found {n_splits}."
    )

group_cv = GroupKFold(n_splits=n_splits)

elastic_cv = cross_validate(
    elastic_model,
    X_train,
    y_train,
    groups=train_df[GROUP],
    cv=group_cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
    n_jobs=-1,
    return_train_score=False,
)

elastic_results = pd.DataFrame({
    "fold": np.arange(1, n_splits + 1),
    "MAE": -elastic_cv["test_mae"],
    "RMSE": -elastic_cv["test_rmse"],
    "R2": elastic_cv["test_r2"],
})

display(elastic_results)

print("ElasticNet grouped CV mean:")
print("MAE :", elastic_results["MAE"].mean())
print("RMSE:", elastic_results["RMSE"].mean())
print("R2  :", elastic_results["R2"].mean())


In [ ]:

# ============================================================
# 14. FIT FINAL ELASTIC NET AND EXTRACT ACTUAL WEIGHTS
# ============================================================
elastic_model.fit(X_train, y_train)

coef = elastic_model.named_steps["model"].coef_

weights_df = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "standardized_weight": coef,
})

weights_df["absolute_weight"] = weights_df["standardized_weight"].abs()
weights_df = weights_df.sort_values(
    "absolute_weight",
    ascending=False
).reset_index(drop=True)

print("Learned ElasticNet weights:")
display(weights_df)

print("Intercept:", elastic_model.named_steps["model"].intercept_)



## 15. Cross-validation / benchmark model — Random Forest

Random Forest is deliberately different from ElasticNet:

- ElasticNet learns a mostly linear relationship and exposes coefficients.
- Random Forest learns nonlinear relationships and feature interactions.
- Comparing them gives a stronger validation of whether the observed relationship is robust to model choice.

This is **not** just another metric. It is an independent model family used as a benchmark.


In [ ]:

# ============================================================
# 15. RANDOM FOREST REGRESSOR — INDEPENDENT MODEL
# ============================================================
rf_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

rf_cv = cross_validate(
    rf_model,
    X_train,
    y_train,
    groups=train_df[GROUP],
    cv=group_cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
    n_jobs=-1,
    return_train_score=False,
)

rf_results = pd.DataFrame({
    "fold": np.arange(1, n_splits + 1),
    "MAE": -rf_cv["test_mae"],
    "RMSE": -rf_cv["test_rmse"],
    "R2": rf_cv["test_r2"],
})

display(rf_results)

print("Random Forest grouped CV mean:")
print("MAE :", rf_results["MAE"].mean())
print("RMSE:", rf_results["RMSE"].mean())
print("R2  :", rf_results["R2"].mean())


In [ ]:

# ============================================================
# 16. MODEL COMPARISON
# ============================================================
comparison = pd.DataFrame([
    {
        "Model": "ElasticNet",
        "CV_MAE": elastic_results["MAE"].mean(),
        "CV_RMSE": elastic_results["RMSE"].mean(),
        "CV_R2": elastic_results["R2"].mean(),
    },
    {
        "Model": "RandomForest",
        "CV_MAE": rf_results["MAE"].mean(),
        "CV_RMSE": rf_results["RMSE"].mean(),
        "CV_R2": rf_results["R2"].mean(),
    },
])

display(comparison.sort_values("CV_MAE"))


In [ ]:

# ============================================================
# 17. FINAL HELD-OUT TEST EVALUATION
# ============================================================
rf_model.fit(X_train, y_train)

elastic_pred = elastic_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

test_results = pd.DataFrame([
    {
        "Model": "ElasticNet",
        "MAE": mean_absolute_error(y_test, elastic_pred),
        "RMSE": mean_squared_error(y_test, elastic_pred, squared=False),
        "R2": r2_score(y_test, elastic_pred),
    },
    {
        "Model": "RandomForest",
        "MAE": mean_absolute_error(y_test, rf_pred),
        "RMSE": mean_squared_error(y_test, rf_pred, squared=False),
        "R2": r2_score(y_test, rf_pred),
    },
])

display(test_results)


In [ ]:

# ============================================================
# 18. RANDOM FOREST FEATURE IMPORTANCE
# ============================================================
rf_importance = rf_model.named_steps["model"].feature_importances_

rf_importance_df = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "importance": rf_importance,
}).sort_values("importance", ascending=False)

display(rf_importance_df)

plt.figure(figsize=(9, 5))
plt.barh(
    rf_importance_df["feature"][::-1],
    rf_importance_df["importance"][::-1],
)
plt.xlabel("Random Forest feature importance")
plt.title("Speech-feature importance")
plt.tight_layout()
plt.show()



## 19. Secondary classification task — Low / Medium / High

The dataset also provides `target_level`.

We treat this as a **secondary classification experiment**. The same anti-leakage rule applies: `target_level` itself and the component assessment scores are not used as predictors.

The second classifier provides an additional independent check of whether the measurable speech variables contain useful information about proficiency level.


In [ ]:

# ============================================================
# 19. CLASSIFICATION DATA
# ============================================================
class_df = df[
    [GROUP] + MODEL_FEATURES + ["target_level"]
].copy()

class_df = class_df.dropna(
    subset=[GROUP, "target_level"]
).reset_index(drop=True)

for c in MODEL_FEATURES:
    class_df[c] = pd.to_numeric(class_df[c], errors="coerce")

class_df = class_df.replace([np.inf, -np.inf], np.nan)

print("Classification rows:", len(class_df))
print("Classes:")
print(class_df["target_level"].value_counts())


In [ ]:

# ============================================================
# 20. SPEAKER-INDEPENDENT CLASSIFICATION SPLIT
# ============================================================
train_idx_c, test_idx_c = next(
    gss.split(
        class_df[MODEL_FEATURES],
        class_df["target_level"],
        groups=class_df[GROUP],
    )
)

train_c = class_df.iloc[train_idx_c].reset_index(drop=True)
test_c = class_df.iloc[test_idx_c].reset_index(drop=True)

X_train_c = train_c[MODEL_FEATURES]
y_train_c = train_c["target_level"]

X_test_c = test_c[MODEL_FEATURES]
y_test_c = test_c["target_level"]

print("Classification train:", len(train_c))
print("Classification test:", len(test_c))
print(
    "Speaker leakage:",
    len(
        set(train_c[GROUP].astype(str))
        .intersection(set(test_c[GROUP].astype(str)))
    ),
)


In [ ]:

# ============================================================
# 21. LOGISTIC REGRESSION + RANDOM FOREST CLASSIFICATION
# ============================================================
logistic_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=RANDOM_SEED,
    )),
])

rf_classifier = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        class_weight="balanced",
    )),
])

class_splits = min(5, train_c[GROUP].nunique())

if class_splits >= 3:
    class_cv = GroupKFold(n_splits=class_splits)

    for name, model in [
        ("LogisticRegression", logistic_model),
        ("RandomForestClassifier", rf_classifier),
    ]:
        scores = cross_validate(
            model,
            X_train_c,
            y_train_c,
            groups=train_c[GROUP],
            cv=class_cv,
            scoring={
                "accuracy": "accuracy",
                "f1_macro": "f1_macro",
            },
            n_jobs=-1,
        )

        print("\n", name)
        print("CV accuracy:", scores["test_accuracy"].mean())
        print("CV macro-F1:", scores["test_f1_macro"].mean())
else:
    print("Not enough unique speakers for 3-fold grouped classification CV.")


In [ ]:

# ============================================================
# 22. FINAL CLASSIFICATION TEST
# ============================================================
logistic_model.fit(X_train_c, y_train_c)
rf_classifier.fit(X_train_c, y_train_c)

pred_log = logistic_model.predict(X_test_c)
pred_rf_c = rf_classifier.predict(X_test_c)

classification_results = pd.DataFrame([
    {
        "Model": "LogisticRegression",
        "Accuracy": accuracy_score(y_test_c, pred_log),
        "Macro_F1": f1_score(y_test_c, pred_log, average="macro"),
    },
    {
        "Model": "RandomForestClassifier",
        "Accuracy": accuracy_score(y_test_c, pred_rf_c),
        "Macro_F1": f1_score(y_test_c, pred_rf_c, average="macro"),
    },
])

display(classification_results)

print("Random Forest classification report:")
print(classification_report(y_test_c, pred_rf_c))


In [ ]:

# ============================================================
# 23. SAVE THE TRAINED MODELS AND WEIGHTS
# ============================================================
OUTPUT_DIR = Path("interviewsense_model_output")
OUTPUT_DIR.mkdir(exist_ok=True)

joblib.dump(
    elastic_model,
    OUTPUT_DIR / "interviewsense_elasticnet_regressor.joblib",
)

joblib.dump(
    rf_model,
    OUTPUT_DIR / "interviewsense_randomforest_regressor.joblib",
)

joblib.dump(
    logistic_model,
    OUTPUT_DIR / "interviewsense_logistic_classifier.joblib",
)

joblib.dump(
    rf_classifier,
    OUTPUT_DIR / "interviewsense_randomforest_classifier.joblib",
)

weights_df.to_csv(
    OUTPUT_DIR / "elasticnet_feature_weights.csv",
    index=False,
)

rf_importance_df.to_csv(
    OUTPUT_DIR / "randomforest_feature_importance.csv",
    index=False,
)

comparison.to_csv(
    OUTPUT_DIR / "regression_cv_comparison.csv",
    index=False,
)

test_results.to_csv(
    OUTPUT_DIR / "regression_test_results.csv",
    index=False,
)

classification_results.to_csv(
    OUTPUT_DIR / "classification_test_results.csv",
    index=False,
)

print("Saved model artifacts to:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


In [ ]:

# ============================================================
# 24. FINAL SANITY CHECK
# ============================================================
print("=" * 70)
print("INTERVIEWSENSE MODELING PIPELINE COMPLETE")
print("=" * 70)

print("Training speakers:", len(train_speakers))
print("Testing speakers:", len(test_speakers))
print("Regression features:", len(MODEL_FEATURES))

print("\nMain interpretable model: ElasticNet")
print("Independent regression benchmark: RandomForestRegressor")
print("Secondary classification models: LogisticRegression + RandomForestClassifier")

print("\nFeature weights:")
display(weights_df[["feature", "standardized_weight"]])

print("\nRegression test results:")
display(test_results)

print("\nClassification test results:")
display(classification_results)



# What this notebook now gives the project

### 1. Actual learned weights
The ElasticNet model produces a coefficient for every measurable speech feature. These are the learned parameters of the interpretable model.

### 2. Independent model validation
Random Forest is trained separately and evaluated with the same speaker-independent folds. If both models perform similarly, that is stronger evidence that the relationship is not merely an artifact of one model family.

### 3. Speaker-independent validation
The same speaker is never allowed to appear in both training and validation/test groups.

### 4. Communication-oriented variables
The modeling phase now includes speaking rate, word count, duration and pause behavior rather than relying on NISQA's acoustic-quality MOS.

### 5. Important limitation
This dataset does **not** provide a direct human-labeled interview communication score covering grammar, semantic relevance, vocabulary, answer quality and reasoning. Therefore this notebook establishes a supervised English-speaking/proficiency model, not a complete interview-intelligence model.

For the final InterviewSense system, the next layer should combine these speech metrics with transcript/ASR features such as filler-word rate, lexical richness, grammar, semantic relevance to the interview question, and answer structure, ideally with human interview/communication ratings.
